# 设备名称匹配

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 配置matplotlib中文显示
import platform
from rapidfuzz import fuzz, process
from sklearn.cluster import AgglomerativeClustering
system_name = platform.system()

if system_name == 'Windows':
    # Windows系统
    plt.rcParams['font.family'] = ['SimHei']
elif system_name == 'Darwin':
    # macOS系统
    plt.rcParams['font.family'] = ['PingFang HK']
elif system_name == 'Linux':
    # Linux系统（可能需要根据具体发行版进行调整）
    plt.rcParams['font.family'] = ['DejaVu Sans']
else:
    # 其他系统或无法识别系统，默认字体
    plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 导入数据

## 油耗表

In [2]:
# import data
file_path = 'Resources/Fuel_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['油耗信息', '设备信息']


In [3]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1

,日期,班次,设备名称,设备编号,油品种类,油品消耗
0,2019-08-04,Night,TEREX TR100#56,HT0056,IC IC,476
1,2019-08-04,Night,TEREX TR100#58,HT0058,IC IC,462
2,2019-08-04,Night,TEREX TR100#59,HT0059,IC IC,451
3,2019-08-04,Night,TEREX TR100#60,HT0060,IC IC,454
4,2019-08-04,Night,TEREX TR100#61,HT0061,IC IC,502
...,...,...,...,...,...,...
222747,2026-04-30,Night,\nTOYOTA LANDCRUISER78 LV#1225,LV1225,Золотой /300007/,60
222748,2026-04-30,Night,9301УБР/ LV#1226 /TOYOTA /LAND 78/,LV1226,Золотой /300007/,30.84
222749,2026-04-30,Night,North Benz ST#137 /ST0137/,ST0137,Золотой /300007/,173.35
222750,2026-04-30,Night,North Benz ST#141,ST141,Золотой /300007/,350


In [4]:
print(df_sheet1["油品消耗"].isnull().sum())
df_sheet1.dropna(inplace=True, subset=["油品消耗"])
print(df_sheet1["油品消耗"].isnull().sum())

1
0


In [5]:
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
df_sheet1['设备编号'] = df_sheet1['设备编号'].str.strip()
# df_sheet1.drop_duplicates(inplace=True)
df_sheet1['日期'] = pd.to_datetime(df_sheet1['日期'])
result_df = df_sheet1.groupby(['设备名称', '设备编号']).agg(
    最早出现日期=('日期', 'min'),
    最晚出现日期=('日期', 'max'),
    出现次数=('设备名称', 'count')
).reset_index() # 重置索引，让设备名称和编号变回普通的列

# 查看结果
result_df

,设备名称,设备编号,最早出现日期,最晚出现日期,出现次数
0,/ LV#1231 /TOYOTA / LAND 79,LV#1231,2025-07-06,2026-04-30,63
1,1958УБП/ LV#1227 /TOYOTA / LAND 78/,LV1227,2025-06-02,2026-04-30,147
2,2220 УЕС SV173,SV173,2025-11-08,2026-04-30,47
3,2659УР/ ME#143 /HELI / CPCD100/,ME#143,2025-10-07,2026-04-27,23
4,3667УКН/ SV#178 /HYUNDAI / AEROSPACE/,SV#178,2026-03-04,2026-04-30,15
...,...,...,...,...,...
504,ZaMine LV-176 /1331/,LV0176,2024-01-02,2026-04-30,366
505,sv130 /CONT/,SV130,2022-03-04,2025-08-07,261
506,Автобус,SV-136,2019-09-05,2019-09-29,3
507,Трейлер,76-75УНЛ,2019-09-03,2019-09-03,1


In [6]:
match_table = pd.DataFrame()
match_table['设备名称'] = result_df['设备名称']
match_table['设备编号'] = result_df['设备编号']
match_table['公司'] = np.nan
match_table['最早出现日期'] = result_df['最早出现日期']
match_table['最晚出现日期'] = result_df['最晚出现日期']
match_table['出现次数'] = result_df['出现次数']
match_table['来源'] = f"Fuel_{xlsx.sheet_names[0]}"
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,最早出现日期,最晚出现日期,出现次数,来源
0,/ LV#1231 /TOYOTA / LAND 79,LV#1231,NaN,2025-07-06,2026-04-30,63,Fuel_油耗信息
1,1958УБП/ LV#1227 /TOYOTA / LAND 78/,LV1227,NaN,2025-06-02,2026-04-30,147,Fuel_油耗信息
2,2220 УЕС SV173,SV173,NaN,2025-11-08,2026-04-30,47,Fuel_油耗信息
3,2659УР/ ME#143 /HELI / CPCD100/,ME#143,NaN,2025-10-07,2026-04-27,23,Fuel_油耗信息
4,3667УКН/ SV#178 /HYUNDAI / AEROSPACE/,SV#178,NaN,2026-03-04,2026-04-30,15,Fuel_油耗信息
...,...,...,...,...,...,...,...
504,ZaMine LV-176 /1331/,LV0176,NaN,2024-01-02,2026-04-30,366,Fuel_油耗信息
505,sv130 /CONT/,SV130,NaN,2022-03-04,2025-08-07,261,Fuel_油耗信息
506,Автобус,SV-136,NaN,2019-09-05,2019-09-29,3,Fuel_油耗信息
507,Трейлер,76-75УНЛ,NaN,2019-09-03,2019-09-03,1,Fuel_油耗信息


In [7]:
df_sheet1 = pd.read_excel(file_path, sheet_name=1)
df_sheet1

,日期,班次,设备名称,设备编号,发动机小时数开始,发动机小时数结束,运行小时数
0,2019-09-01,Day,TEREX TR100#58,HT0058,301.1,311,9.9
1,2019-09-01,Day,TEREX TR100#64,HT0064,307.7,307.7,0
2,2019-09-01,Day,NHL NTE240 #66,HT0066,64,64,0
3,2019-09-01,Day,NHL NTE240 #67,HT0067,62,62,0
4,2019-09-01,Day,NHL NTE240 #68,HT0068,66,66,0
...,...,...,...,...,...,...,...
627207,2026-04-30,Night,XDM100 XD#3130 /RENT/,XD#3130/1173,3632,3632,0
627208,2026-04-30,Night,XD#3133/1176,XD#3133/1176,8821,8821,0
627209,2026-04-30,Night,XDE120 XD#3134 /RENT/,XD#3134/1177,10234.8,10234.8,0
627210,2026-04-30,Night,XDE130 XD#3136 /RENT/,XD#3136/1179,3329.4,3329.4,0


In [8]:
print(df_sheet1["设备名称"].isnull().sum())
# df_sheet1.dropna(inplace=True, subset=["油品消耗"])
# print(df_sheet1["油品消耗"].isnull().sum())

62


In [9]:
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
df_sheet1['设备编号'] = df_sheet1['设备编号'].str.strip()

df_sheet1['日期'] = pd.to_datetime(df_sheet1['日期'])
result_df = df_sheet1.groupby(['设备名称', '设备编号']).agg(
    最早出现日期=('日期', 'min'),
    最晚出现日期=('日期', 'max'),
    出现次数=('设备名称', 'count')
).reset_index() # 重置索引，让设备名称和编号变回普通的列
result_df['来源'] = f"Fuel_{xlsx.sheet_names[1]}"

# 查看结果
result_df
# df_sheet1.drop_duplicates(inplace=True)

,设备名称,设备编号,最早出现日期,最晚出现日期,出现次数,来源
0,/ LV#1231 /TOYOTA / LAND 79,LV#1231,2025-07-06,2026-04-30,597,Fuel_设备信息
1,1958УБП/ LV#1227 /TOYOTA / LAND 78/,LV1227,2025-06-01,2026-04-30,668,Fuel_设备信息
2,2220 УЕС SV173,SV173,2025-11-08,2026-04-30,348,Fuel_设备信息
3,2659УР/ ME#143 /HELI / CPCD100/,ME#143,2025-10-07,2026-04-30,411,Fuel_设备信息
4,3667УКН/ SV#178 /HYUNDAI / AEROSPACE/,SV#178,2026-03-01,2026-04-30,122,Fuel_设备信息
...,...,...,...,...,...,...
497,ZaMine LV-176,LV0176,2019-11-01,2023-12-31,1517,Fuel_设备信息
498,ZaMine LV-176 /1331/,LV0176,2024-01-01,2026-04-30,1706,Fuel_设备信息
499,sv130 /CONT/,SV130,2022-03-01,2026-04-30,3045,Fuel_设备信息
500,Автобус,SV-136,2019-09-05,2019-09-30,51,Fuel_设备信息


In [10]:
match_table = pd.concat([match_table, result_df], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,最早出现日期,最晚出现日期,出现次数,来源
0,/ LV#1231 /TOYOTA / LAND 79,LV#1231,NaN,2025-07-06,2026-04-30,63,Fuel_油耗信息
1,1958УБП/ LV#1227 /TOYOTA / LAND 78/,LV1227,NaN,2025-06-02,2026-04-30,147,Fuel_油耗信息
2,2220 УЕС SV173,SV173,NaN,2025-11-08,2026-04-30,47,Fuel_油耗信息
3,2659УР/ ME#143 /HELI / CPCD100/,ME#143,NaN,2025-10-07,2026-04-27,23,Fuel_油耗信息
4,3667УКН/ SV#178 /HYUNDAI / AEROSPACE/,SV#178,NaN,2026-03-04,2026-04-30,15,Fuel_油耗信息
...,...,...,...,...,...,...,...
1006,ZaMine LV-176,LV0176,NaN,2019-11-01,2023-12-31,1517,Fuel_设备信息
1007,ZaMine LV-176 /1331/,LV0176,NaN,2024-01-01,2026-04-30,1706,Fuel_设备信息
1008,sv130 /CONT/,SV130,NaN,2022-03-01,2026-04-30,3045,Fuel_设备信息
1009,Автобус,SV-136,NaN,2019-09-05,2019-09-30,51,Fuel_设备信息


## 电力表

In [11]:
# import data
file_path = 'Resources/电力_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['电力消耗']


In [12]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1

,日期,设备名称,电力消耗
0,2023-02-01,EX-5600#111,20873.9552
1,2023-02-01,EX-5600#113,26296.6428
2,2023-02-02,EX-5600#111,18047.4620
3,2023-02-02,EX-5600#113,25303.8492
4,2023-02-03,EX-5600#113,26525.0880
...,...,...,...
5262,2026-04-30,EX-5600#113,24600.0000
5263,2026-04-30,EX-5600#122,26892.0000
5264,2026-04-30,EX-3600#112,20268.0000
5265,2026-04-30,EX-5600#111,26568.0000


In [13]:
print(df_sheet1["电力消耗"].isnull().sum())
df_sheet1.dropna(inplace=True, subset=["电力消耗"])
print(df_sheet1["电力消耗"].isnull().sum())

0
0


In [14]:
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
# df_sheet1['设备编号'] = np.nan
df_sheet1['日期'] = pd.to_datetime(df_sheet1['日期'])
df_sheet1

,日期,设备名称,电力消耗
0,2023-02-01,EX-5600#111,20873.9552
1,2023-02-01,EX-5600#113,26296.6428
2,2023-02-02,EX-5600#111,18047.4620
3,2023-02-02,EX-5600#113,25303.8492
4,2023-02-03,EX-5600#113,26525.0880
...,...,...,...
5262,2026-04-30,EX-5600#113,24600.0000
5263,2026-04-30,EX-5600#122,26892.0000
5264,2026-04-30,EX-3600#112,20268.0000
5265,2026-04-30,EX-5600#111,26568.0000


In [15]:
# df_sheet1.drop_duplicates(inplace=True)
result_df = df_sheet1.groupby(['设备名称']).agg(
    最早出现日期=('日期', 'min'),
    最晚出现日期=('日期', 'max'),
    出现次数=('设备名称', 'count')
).reset_index() # 重置索引，让设备名称和编号变回普通的列
result_df['来源'] = f"电力_{xlsx.sheet_names[0]}"
result_df

,设备名称,最早出现日期,最晚出现日期,出现次数,来源
0,EX-3600#112,2023-04-07,2026-04-30,842,电力_电力消耗
1,EX-5600#111,2023-02-01,2026-04-30,1096,电力_电力消耗
2,EX-5600#113,2023-02-01,2026-04-30,1110,电力_电力消耗
3,EX-5600#122,2023-02-04,2026-04-30,1167,电力_电力消耗
4,EX-5600#123,2023-03-01,2026-04-30,1052,电力_电力消耗


In [16]:
match_table = pd.concat([match_table, result_df], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,最早出现日期,最晚出现日期,出现次数,来源
0,/ LV#1231 /TOYOTA / LAND 79,LV#1231,NaN,2025-07-06,2026-04-30,63,Fuel_油耗信息
1,1958УБП/ LV#1227 /TOYOTA / LAND 78/,LV1227,NaN,2025-06-02,2026-04-30,147,Fuel_油耗信息
2,2220 УЕС SV173,SV173,NaN,2025-11-08,2026-04-30,47,Fuel_油耗信息
3,2659УР/ ME#143 /HELI / CPCD100/,ME#143,NaN,2025-10-07,2026-04-27,23,Fuel_油耗信息
4,3667УКН/ SV#178 /HYUNDAI / AEROSPACE/,SV#178,NaN,2026-03-04,2026-04-30,15,Fuel_油耗信息
...,...,...,...,...,...,...,...
1011,EX-3600#112,NaN,NaN,2023-04-07,2026-04-30,842,电力_电力消耗
1012,EX-5600#111,NaN,NaN,2023-02-01,2026-04-30,1096,电力_电力消耗
1013,EX-5600#113,NaN,NaN,2023-02-01,2026-04-30,1110,电力_电力消耗
1014,EX-5600#122,NaN,NaN,2023-02-04,2026-04-30,1167,电力_电力消耗


## 产量

In [17]:
# import data
file_path = 'Resources/产量_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['生产数据', '运行数据']


In [18]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1

,日期,班次,矿卡名称,挖机名称,矿石类型,数量,产量
0,2019-08-17,Day,TEREX TR100 #56,CAT 390D #02,Хө,4.0,128.0
1,2019-08-17,Day,TEREX TR100 #57,CAT 390D #02,Хө,4.0,128.0
2,2019-08-17,Day,TEREX TR100 #58,CAT 390D #02,Хө,3.0,96.0
3,2019-08-17,Day,TEREX TR100 #59,CAT 390D #02,Хө,4.0,128.0
4,2019-08-17,Day,TEREX TR100 #60,CAT 390D #02,Хө,5.0,160.0
...,...,...,...,...,...,...,...
337392,2026-04-30,Night,NHL NTE240AC #1220,EX5600-6 #113,Хө,19.0,1615.0
337393,2026-04-30,Night,NHL NTE240AC #1221,EX5600-6 #113,Хө,19.0,1615.0
337394,2026-04-30,Night,NHL NTE240AC #1221,EX2600-6 #121,Хө,1.0,85.0
337395,2026-04-30,Night,NHL NTE240AC #1222,EX5600-6 #113,Хө,18.0,1530.0


In [19]:
data_to_process = df_sheet1[(df_sheet1["产量"]==0)&(df_sheet1["数量"]!=0)]
data_to_process

,日期,班次,矿卡名称,挖机名称,矿石类型,数量,产量
224415,2025-01-09,Night,BUCYRUS MT4400AC #1017,EX2600-6 #120,Хө,2.0,0.0
227512,2025-01-21,Day,BUCYRUS MT4400AC #1019,XE2000 EX#308/149,Хө,1.0,0.0
227513,2025-01-21,Day,BUCYRUS MT4400AC #1020,XE2000 EX#308/149,Хө,2.0,0.0
229737,2025-01-29,Day,BUCYRUS MT4400AC #1018,EX3600-6#112,Хө,2.0,0.0
229738,2025-01-29,Day,BUCYRUS MT4400AC #1018,EX2600-6 #120,Хө,10.0,0.0
229739,2025-01-29,Day,BUCYRUS MT4400AC #1019,EX3600-6#112,Хө,2.0,0.0
229740,2025-01-29,Day,BUCYRUS MT4400AC #1019,EX2600-6 #120,Хө,10.0,0.0
230215,2025-01-30,Night,BUCYRUS MT4400AC #1016,EX5600-6 #122,Хө,4.0,0.0
230216,2025-01-30,Night,BUCYRUS MT4400AC #1017,EX5600-6 #122,Хө,3.0,0.0
230217,2025-01-30,Night,BUCYRUS MT4400AC #1018,EX5600-6 #122,Хө,5.0,0.0


In [20]:
data_to_process.drop_duplicates(inplace=True)
data_to_process.to_excel("resources/产量_处理.xlsx", index=False)

In [21]:
print(df_sheet1["数量"].isnull().sum())
df_sheet1.dropna(inplace=True, subset=["数量"])
print(df_sheet1["数量"].isnull().sum())

0
0


In [22]:
df = pd.DataFrame()
df['设备名称'] = df_sheet1['挖机名称']
df['日期']=df_sheet1['日期']
df['来源'] = f"产量_{xlsx.sheet_names[0]}"
df2 = pd.DataFrame()
df2['设备名称'] = df_sheet1['矿卡名称']
df2['日期']=df_sheet1['日期']
df2['来源'] = f"产量_{xlsx.sheet_names[1]}"
df = pd.concat([df, df2], ignore_index=True)

result_df = df.groupby(['设备名称']).agg(
    来源=('来源', 'first'),
    最早出现日期=('日期', 'min'),
    最晚出现日期=('日期', 'max'),
    出现次数=('设备名称', 'count')
).reset_index() # 重置索引，让设备名称和编号变回普通的列

# 查看结果
result_df
# df.drop_duplicates(inplace=True)

,设备名称,来源,最早出现日期,最晚出现日期,出现次数
0,#101,产量_生产数据,2024-04-03,2024-04-03,3
1,#103,产量_生产数据,2023-10-12,2024-11-08,32
2,#104,产量_生产数据,2023-08-09,2025-12-05,27
3,#105,产量_生产数据,2023-10-12,2024-01-20,24
4,#106,产量_生产数据,2023-10-12,2025-03-20,67
...,...,...,...,...,...
889,装载数\nHitachi 2600#121 /CHU/,产量_生产数据,2021-01-19,2021-02-28,819
890,装载数 \nHitachi 2600#119 /CHU/,产量_生产数据,2021-01-18,2021-02-25,96
891,装载数 \nHitachi 2600#120 /CHU/,产量_生产数据,2021-01-18,2021-02-25,51
892,装载数 \nHitachi 2600#121 /CHU/,产量_生产数据,2021-01-18,2021-02-25,87


In [23]:
match_table = pd.concat([match_table, result_df], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,最早出现日期,最晚出现日期,出现次数,来源
0,/ LV#1231 /TOYOTA / LAND 79,LV#1231,NaN,2025-07-06,2026-04-30,63,Fuel_油耗信息
1,1958УБП/ LV#1227 /TOYOTA / LAND 78/,LV1227,NaN,2025-06-02,2026-04-30,147,Fuel_油耗信息
2,2220 УЕС SV173,SV173,NaN,2025-11-08,2026-04-30,47,Fuel_油耗信息
3,2659УР/ ME#143 /HELI / CPCD100/,ME#143,NaN,2025-10-07,2026-04-27,23,Fuel_油耗信息
4,3667УКН/ SV#178 /HYUNDAI / AEROSPACE/,SV#178,NaN,2026-03-04,2026-04-30,15,Fuel_油耗信息
...,...,...,...,...,...,...,...
1905,装载数\nHitachi 2600#121 /CHU/,NaN,NaN,2021-01-19,2021-02-28,819,产量_生产数据
1906,装载数 \nHitachi 2600#119 /CHU/,NaN,NaN,2021-01-18,2021-02-25,96,产量_生产数据
1907,装载数 \nHitachi 2600#120 /CHU/,NaN,NaN,2021-01-18,2021-02-25,51,产量_生产数据
1908,装载数 \nHitachi 2600#121 /CHU/,NaN,NaN,2021-01-18,2021-02-25,87,产量_生产数据


## 效率

In [24]:
# import data
file_path = 'Resources/效率_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['Sheet1']


In [25]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)

In [26]:
df_sheet1

,日期,班次,序号 Д/д,设备种类 Техникийн төрөл,公司 Компани,应运行分钟 Ажиллах мин,应运行小时数 Ажиллах мот цаг,停车/换班/\nСул Зогсолт /Ээлж солигдох /,转移 Шилжих\nхөдөлгөөн,挖机场地推土 /清理墙壁/\nУл талбай түрүүлэх/Ханын цэвэрлэгээ/,...,未计划/故障 Төлөвлөгөөт бус/эвдрэл,待命 Бэлэн байдалд,因天气：大风暴，雨，雪Цаг агаар: Хүчтэй салхи. Бороо.Цас,扬尘：洒水车不足 Тоосжилт: Усны машин хангалтгүй,排队/装水/ Дараалалд зогсох /Ус дүүргэх/,总产量生产运行分钟 Нийт бүтээлтэй ажилласан мин,因电力原因停车。/计划/Цахилгаанаас шалтгаалсан зогсолт ./Төлөвлөсөн/,因电力原因停车。/未计划/Цахилгаанаас шалтгаалсан зогсолт /Төлөвлөгөөт бус/,总产量生产运行小时 Нийт бүтээлтэй ажилласан цаг,注释 Тайлбар
0,2019-08-01,Day,1,LIEBHERR R996B #07,NaN,720,12,60,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,570,NaN,NaN,9.5,NaN
1,2019-08-01,Day,2,HITACHI EX3600 #01KH,NaN,720,12,60,30,30,...,NaN,30,NaN,NaN,NaN,510,NaN,NaN,8.5,NaN
2,2019-08-01,Day,3,HITACHI EX2600E #08,NaN,720,12,60,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,600,NaN,NaN,10,NaN
3,2019-08-01,Day,4,HITACHI EX2600E #03KH,NaN,720,12,60,210,20,...,NaN,NaN,NaN,NaN,NaN,370,NaN,NaN,6.166667,NaN
4,2019-08-01,Day,5,LIEBHERR R9350 #06,NaN,720,12,70,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,580,NaN,NaN,9.666667,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416409,2026-04-30,Night,142,XDE120 XD#1174,Normount,720,12,NaN,NaN,NaN,...,720,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN
416410,2026-04-30,Night,143,XDE120 XD#1175,Normount,720,12,NaN,NaN,NaN,...,720,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN
416411,2026-04-30,Night,144,XDE120 XD#1176,Normount,720,12,NaN,NaN,NaN,...,720,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN
416412,2026-04-30,Night,145,XDE120 XD#1177,Normount,720,12,NaN,NaN,NaN,...,720,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN


In [27]:
df_sheet1["设备种类                             Техникийн төрөл "].value_counts()

设备种类                             Техникийн төрөл 
CAT D8T #168                                         3156
CAT D8T #169                                         3156
CAT D8T #172                                         3156
CAT D8T #170                                         3152
CAT D8T #171                                         3152
                                                     ... 
NTE240#1220                                             1
NTE240#1221                                             1
NTE240#1222                                             1
设备种类                             Техникийн төрөл        1
WT#1212                                                 1
Name: count, Length: 876, dtype: int64

In [28]:
# 查看第4列
df_sheet1['设备名称'] = df_sheet1.iloc[:, 3].str.strip()
df_sheet1['公司'] = df_sheet1.iloc[:, 4].str.strip()
df_sheet1['日期'] = pd.to_datetime(df_sheet1['日期'])
df = df_sheet1[['日期','设备名称','公司']]
df

,日期,设备名称,公司
0,2019-08-01,LIEBHERR R996B #07,NaN
1,2019-08-01,HITACHI EX3600 #01KH,NaN
2,2019-08-01,HITACHI EX2600E #08,NaN
3,2019-08-01,HITACHI EX2600E #03KH,NaN
4,2019-08-01,LIEBHERR R9350 #06,NaN
...,...,...,...
416409,2026-04-30,XDE120 XD#1174,Normount
416410,2026-04-30,XDE120 XD#1175,Normount
416411,2026-04-30,XDE120 XD#1176,Normount
416412,2026-04-30,XDE120 XD#1177,Normount


In [29]:
# df.drop_duplicates(inplace=True)
result_df = df.groupby(['设备名称']).agg(
    公司=('公司', 'first'),
    最早出现日期=('日期', 'min'),
    最晚出现日期=('日期', 'max'),
    出现次数=('设备名称', 'count')
).reset_index() # 重置索引，让设备名称和编号变回普通的列
result_df['来源'] = f"效率_{xlsx.sheet_names[0]}"

# 查看结果
result_df

,设备名称,公司,最早出现日期,最晚出现日期,出现次数,来源
0,834#173,Normount,2022-02-10,2022-02-28,19,效率_Sheet1
1,834K #173,Normount,2022-02-08,2022-02-09,2,效率_Sheet1
2,BELAZ 75135 #117 /OLB/,NaN,2019-08-17,2019-11-29,39,效率_Sheet1
3,BELAZ 75135 #117OLB,NaN,2019-08-01,2019-10-05,34,效率_Sheet1
4,BELAZ 75135 #118 /OLB/,NaN,2019-08-17,2019-11-29,39,效率_Sheet1
...,...,...,...,...,...,...
857,XS183J #02 /19-р товчоо/,NaN,2019-11-25,2019-11-29,8,效率_Sheet1
858,ZL50GN Lo #146,Normount,2023-02-03,2023-03-19,28,效率_Sheet1
859,Шөнө,NaN,2022-01-09,2022-01-31,23,效率_Sheet1
860,夜班 Шөнө,NaN,2022-04-01,2026-04-19,152,效率_Sheet1


In [30]:
match_table = pd.concat([match_table, result_df], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,最早出现日期,最晚出现日期,出现次数,来源
0,/ LV#1231 /TOYOTA / LAND 79,LV#1231,NaN,2025-07-06,2026-04-30,63,Fuel_油耗信息
1,1958УБП/ LV#1227 /TOYOTA / LAND 78/,LV1227,NaN,2025-06-02,2026-04-30,147,Fuel_油耗信息
2,2220 УЕС SV173,SV173,NaN,2025-11-08,2026-04-30,47,Fuel_油耗信息
3,2659УР/ ME#143 /HELI / CPCD100/,ME#143,NaN,2025-10-07,2026-04-27,23,Fuel_油耗信息
4,3667УКН/ SV#178 /HYUNDAI / AEROSPACE/,SV#178,NaN,2026-03-04,2026-04-30,15,Fuel_油耗信息
...,...,...,...,...,...,...,...
2767,XS183J #02 /19-р товчоо/,NaN,NaN,2019-11-25,2019-11-29,8,效率_Sheet1
2768,ZL50GN Lo #146,NaN,Normount,2023-02-03,2023-03-19,28,效率_Sheet1
2769,Шөнө,NaN,NaN,2022-01-09,2022-01-31,23,效率_Sheet1
2770,夜班 Шөнө,NaN,NaN,2022-04-01,2026-04-19,152,效率_Sheet1


In [31]:
df = match_table.copy()
# 按照最晚出现日期排序
df = df.sort_values(by=['设备名称', '最晚出现日期'], na_position='last')
df

,设备名称,设备编号,公司,最早出现日期,最晚出现日期,出现次数,来源
1016,#101,NaN,NaN,2024-04-03,2024-04-03,3,产量_生产数据
1017,#103,NaN,NaN,2023-10-12,2024-11-08,32,产量_生产数据
1018,#104,NaN,NaN,2023-08-09,2025-12-05,27,产量_生产数据
1019,#105,NaN,NaN,2023-10-12,2024-01-20,24,产量_生产数据
1020,#106,NaN,NaN,2023-10-12,2025-03-20,67,产量_生产数据
...,...,...,...,...,...,...,...
1906,装载数 \nHitachi 2600#119 /CHU/,NaN,NaN,2021-01-18,2021-02-25,96,产量_生产数据
1907,装载数 \nHitachi 2600#120 /CHU/,NaN,NaN,2021-01-18,2021-02-25,51,产量_生产数据
1908,装载数 \nHitachi 2600#121 /CHU/,NaN,NaN,2021-01-18,2021-02-25,87,产量_生产数据
1909,装载数 Hitachi 2600#121 /CHU/,NaN,NaN,2021-01-18,2021-02-18,33,产量_生产数据


In [32]:
df.to_excel('Resources/设备名称匹配清单（大全）.xlsx', index=False)

In [33]:
#合并名称
final_result = df.groupby(['设备名称', '设备编号'],dropna=False).agg(
    来源=('来源', 'first'),
    公司=('公司', 'first'),
    最早出现日期=('最早出现日期', 'min'),
    最晚出现日期=('最晚出现日期', 'max'),
    出现次数=('出现次数', 'sum')  # <--- 注意这里变成了求和 (sum)
).reset_index()
final_result = final_result.sort_values(by=['设备名称', '设备编号','出现次数'])
final_result

,设备名称,设备编号,来源,公司,最早出现日期,最晚出现日期,出现次数
0,#101,NaN,产量_生产数据,None,2024-04-03,2024-04-03,3
1,#103,NaN,产量_生产数据,None,2023-10-12,2024-11-08,32
2,#104,NaN,产量_生产数据,None,2023-08-09,2025-12-05,27
3,#105,NaN,产量_生产数据,None,2023-10-12,2024-01-20,24
4,#106,NaN,产量_生产数据,None,2023-10-12,2025-03-20,67
...,...,...,...,...,...,...,...
1783,装载数 \nHitachi 2600#119 /CHU/,NaN,产量_生产数据,None,2021-01-18,2021-02-25,96
1784,装载数 \nHitachi 2600#120 /CHU/,NaN,产量_生产数据,None,2021-01-18,2021-02-25,51
1785,装载数 \nHitachi 2600#121 /CHU/,NaN,产量_生产数据,None,2021-01-18,2021-02-25,87
1786,装载数 Hitachi 2600#121 /CHU/,NaN,产量_生产数据,None,2021-01-18,2021-02-18,33


In [34]:
match_table.to_excel('Resources/设备名称匹配清单.xlsx', index=False)
final_result.to_excel('Resources/设备名称匹配清单（详细）.xlsx', index=False)

# 匹配

## 导入

In [35]:
# import data
# table_to_match_fp = "Resources/设备名称匹配清单.xlsx"
# table_to_match_fp = "Resources/设备名称匹配清单（详细）.xlsx"
table_to_match_fp = "Resources/设备名称匹配清单（大全）.xlsx"
table_to_match = pd.ExcelFile(table_to_match_fp)
print(table_to_match.sheet_names)

['Sheet1']


In [36]:
table_to_match_df = pd.read_excel(table_to_match_fp, sheet_name=0)
table_to_match_df.head()

,设备名称,设备编号,公司,最早出现日期,最晚出现日期,出现次数,来源
0,#101,NaN,NaN,2024-04-03,2024-04-03,3,产量_生产数据
1,#103,NaN,NaN,2023-10-12,2024-11-08,32,产量_生产数据
2,#104,NaN,NaN,2023-08-09,2025-12-05,27,产量_生产数据
3,#105,NaN,NaN,2023-10-12,2024-01-20,24,产量_生产数据
4,#106,NaN,NaN,2023-10-12,2025-03-20,67,产量_生产数据


In [37]:
table_to_match_df.columns

Index(['设备名称', '设备编号', '公司', '最早出现日期', '最晚出现日期', '出现次数', '来源'], dtype='str')

In [38]:
# import data
match_table_fp = "Resources/设备名称表.xlsx"
match_table = pd.ExcelFile(match_table_fp)
print(match_table.sheet_names)

['匹配表']


In [39]:
match_table_df = pd.read_excel(match_table_fp, sheet_name=0)
match_table_df.head()

,设备名称,设备编号,公司,来源,最早出现日期,最晚出现日期,出现次数,标准设备编号,标准设备名称,标准公司名称,匹配情况
0,Шөнө,Шөнө,NaN,效率_Sheet1,2022-01-09,2022-01-31,23,AA#0001,忽略,NaN,1.0
1,设备种类 Техникийн төрөл,设备种类 Техникийн төрөл,公司 Компани,效率_Sheet1,2026-04-19,2026-04-19,1,AA#0001,忽略,NaN,1.0
2,夜班 Шөнө,夜班 Шөнө,NaN,效率_Sheet1,2022-04-01,2026-04-19,152,AA#0001,忽略,NaN,1.0
3,CAT D8 #09,CAT D8 #09,NaN,效率_Sheet1,2019-10-05,2019-10-05,2,AA#0001,忽略,NaN,1.0
4,CAT D8 #10,CAT D8 #10,NaN,效率_Sheet1,2019-10-05,2019-10-05,2,AA#0001,忽略,NaN,1.0


In [40]:
match_table_df.columns

Index(['设备名称', '设备编号', '公司', '来源', '最早出现日期', '最晚出现日期', '出现次数', '标准设备编号',
       '标准设备名称', '标准公司名称', '匹配情况'],
      dtype='str')

# 匹配

In [56]:
import pandas as pd
import re
import difflib

# --- 1. 辅助函数定义 ---

# 字符串相似度计算(0-100)
def get_similarity(s1, s2):
    if pd.isna(s1) or pd.isna(s2):
        return 0
    return difflib.SequenceMatcher(None, str(s1), str(s2)).ratio() * 100

# 提取数字规则
def extract_numbers(text):
    if pd.isna(text):
        return []
    text = str(text)
    # 如果有 #，提取 # 后面的数字
    if '#' in text:
        # 匹配 # 后面的连续数字
        matches = re.findall(r'#(\d+)', text)
        if matches:
            return [int(m) for m in matches]
    # 如果没有 #，或者 # 后面没提取到，提取所有数字片段
    matches = re.findall(r'(\d+)', text)
    return [int(m) for m in matches] if matches else []

# --- 2. 主匹配逻辑函数 ---

def match_equipment(table_to_match_df, match_table_df, threshold=85):
    # 拷贝数据避免修改原始数据
    df_obj = table_to_match_df.copy()
    df_std = match_table_df.copy()

    # 规则1: 缺失值的互相填充
    df_obj['设备名称'] = df_obj['设备名称'].fillna(df_obj['设备编号'])
    df_obj['设备编号'] = df_obj['设备编号'].fillna(df_obj['设备名称'])

    # 为标准表预先提取数字，构建 {提取数字: [标准表行数据]} 的字典以加速搜索
    std_num_dict = {}
    for idx, row in df_std.iterrows():
        std_no = row['标准设备编号']
        nums = extract_numbers(std_no)
        if nums:
            std_n = nums[0] # 标准编号一般只有一个主要数字
            if std_n not in std_num_dict:
                std_num_dict[std_n] = []
            std_num_dict[std_n].append(row)

    # 准备结果列表
    results = []

    # 逐行处理需要匹配的表
    for idx, row in df_obj.iterrows():
        obj_name = row['设备名称']
        obj_no = row['设备编号']

        # 默认结果字段
        match_info = {
            '标准设备编号': None, '标准设备名称': None, '标准公司名称': None,
            '匹配情况': False, '匹配模式': '匹配失败',
            '最高匹配相似度': 0.0, '最高匹配名称': None
        }

        # 规则2: 设备名称精确匹配
        match_by_name = df_std[df_std['设备名称'] == obj_name]
        if not match_by_name.empty:
            std_row = match_by_name.iloc[0]
            match_info.update({
                '标准设备编号': std_row['标准设备编号'], '标准设备名称': std_row['标准设备名称'],
                '标准公司名称': std_row['标准公司名称'], '匹配情况': True, '匹配模式': '规则2-名称精确匹配',
                '最高匹配相似度': 100.0, '最高匹配名称': std_row['标准设备名称']
            })
            results.append({**row.to_dict(), **match_info})
            continue

        # # 规则3: 设备编号精确匹配
        # match_by_no = df_std[df_std['设备编号'] == obj_no]
        # if not match_by_no.empty:
        #     std_row = match_by_no.iloc[0]
        #     match_info.update({
        #         '标准设备编号': std_row['标准设备编号'], '标准设备名称': std_row['标准设备名称'],
        #         '标准公司名称': std_row['标准公司名称'], '匹配情况': True, '匹配模式': '规则3-编号精确匹配',
        #         '最高匹配相似度': 100.0, '最高匹配名称': std_row['标准设备名称']
        #     })
        #     results.append({**row.to_dict(), **match_info})
        #     continue

        # 规则4 & 5 提取数字匹配逻辑的闭包函数
        def fuzzy_match_by_numbers(text_to_extract, mode_name):
            extracted_nums = extract_numbers(text_to_extract)
            best_sim = 0.0
            best_std_row = None
            best_std_name = None

            for num in extracted_nums:
                # 尝试当前数字，如果没匹配到，尝试 + 1000
                search_nums = [num, num + 1000]

                for s_num in search_nums:
                    candidates = std_num_dict.get(s_num, [])
                    for cand in candidates:
                        sim = get_similarity(obj_name, cand['标准设备名称'])
                        if sim > best_sim:
                            best_sim = sim
                            best_std_row = cand
                            best_std_name = cand['标准设备名称']

            if best_sim >= threshold and best_std_row is not None:
                return {
                    '标准设备编号': best_std_row['标准设备编号'],
                    '标准设备名称': best_std_row['标准设备名称'],
                    '标准公司名称': best_std_row['标准公司名称'],
                    '匹配情况': True, '匹配模式': mode_name,
                    '最高匹配相似度': round(best_sim, 2), '最高匹配名称': best_std_name
                }
            elif best_std_row is not None:
                # 记录最高相似度但未成功匹配的状态
                return {
                    '最高匹配相似度': round(best_sim, 2), '最高匹配名称': best_std_name
                }
            return {}

        # 规则4: 设备编号提取数字模糊匹配
        rule4_res = fuzzy_match_by_numbers(obj_no, '规则4-设备编号模糊匹配')
        if rule4_res.get('匹配情况'):
            match_info.update(rule4_res)
            results.append({**row.to_dict(), **match_info})
            continue
        else:
            match_info.update(rule4_res) # 更新可能的最高相似度信息

        # 规则5: 设备名称提取数字模糊匹配
        rule5_res = fuzzy_match_by_numbers(obj_name, '规则5-设备名称模糊匹配')
        if rule5_res.get('匹配情况'):
            match_info.update(rule5_res)
            results.append({**row.to_dict(), **match_info})
            continue
        elif rule5_res.get('最高匹配相似度', 0) > match_info['最高匹配相似度']:
            match_info.update(rule5_res)

        # 均未匹配成功
        results.append({**row.to_dict(), **match_info})

    # 转换为DataFrame并规范输出列排序
    final_cols = ['设备名称', '设备编号', '公司', '来源','最早出现日期','最晚出现日期','出现次数',
                  '标准设备编号', '标准设备名称',
                  '标准公司名称', '匹配情况', '匹配模式', '最高匹配相似度', '最高匹配名称']
    result_df = pd.DataFrame(results)[final_cols]

    return result_df

In [57]:
# --- 3. 运行测试（你可以用你的DataFrame替换这里的测试数据） ---
result_df = match_equipment(table_to_match_df, match_table_df,30)
result_df

,设备名称,设备编号,公司,来源,最早出现日期,最晚出现日期,出现次数,标准设备编号,标准设备名称,标准公司名称,匹配情况,匹配模式,最高匹配相似度,最高匹配名称
0,#101,#101,NaN,产量_生产数据,2024-04-03,2024-04-03,3,EX#0101,CAT 390D EX#0101,NORMOUNT,True,规则2-名称精确匹配,100.0,CAT 390D EX#0101
1,#103,#103,NaN,产量_生产数据,2023-10-12,2024-11-08,32,EX#0103,LIEBHERR R984 EX#0103,MONNIS,True,规则2-名称精确匹配,100.0,LIEBHERR R984 EX#0103
2,#104,#104,NaN,产量_生产数据,2023-08-09,2025-12-05,27,EX#0104,LIEBHERR R984 EX#0104,MONNIS,True,规则2-名称精确匹配,100.0,LIEBHERR R984 EX#0104
3,#105,#105,NaN,产量_生产数据,2023-10-12,2024-01-20,24,EX#0105,LIEBHERR R984 EX#0105,MONNIS,True,规则2-名称精确匹配,100.0,LIEBHERR R984 EX#0105
4,#106,#106,NaN,产量_生产数据,2023-10-12,2025-03-20,67,EX#0106,LIEBHERR R9350 EX#0106,MONNIS,True,规则2-名称精确匹配,100.0,LIEBHERR R9350 EX#0106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2767,装载数 \nHitachi 2600#119 /CHU/,装载数 \nHitachi 2600#119 /CHU/,NaN,产量_生产数据,2021-01-18,2021-02-25,96,EX#0119,HITACHI EX2600 EX#0119,NORMOUNT,True,规则2-名称精确匹配,100.0,HITACHI EX2600 EX#0119
2768,装载数 \nHitachi 2600#120 /CHU/,装载数 \nHitachi 2600#120 /CHU/,NaN,产量_生产数据,2021-01-18,2021-02-25,51,EX#0120,HITACHI EX2600 EX#0120,NORMOUNT,True,规则2-名称精确匹配,100.0,HITACHI EX2600 EX#0120
2769,装载数 \nHitachi 2600#121 /CHU/,装载数 \nHitachi 2600#121 /CHU/,NaN,产量_生产数据,2021-01-18,2021-02-25,87,EX#0121,HITACHI EX2600 EX#0121,NORMOUNT,True,规则2-名称精确匹配,100.0,HITACHI EX2600 EX#0121
2770,装载数 Hitachi 2600#121 /CHU/,装载数 Hitachi 2600#121 /CHU/,NaN,产量_生产数据,2021-01-18,2021-02-18,33,EX#0121,HITACHI EX2600 EX#0121,NORMOUNT,True,规则2-名称精确匹配,100.0,HITACHI EX2600 EX#0121


In [58]:
result_df["匹配情况"].value_counts()

匹配情况
True    2772
Name: count, dtype: int64

In [59]:
# 过滤掉“设备名称”为纯数字的行
# result_df = result_df[~result_df['设备名称'].str.match(r'^\d+$')]
result_df["匹配情况"].value_counts()

匹配情况
True    2772
Name: count, dtype: int64

In [60]:
result_df.to_excel("resources/result.xlsx", index=False)

# 匹配结果

In [5]:
match_result_path = "Resources/设备名称表.xlsx"
match_result = pd.ExcelFile(match_result_path)
print(match_result.sheet_names)

['匹配表']


In [6]:
match_result_df = pd.read_excel(match_result_path, sheet_name=0)
match_result_df

,设备名称,设备编号,公司,来源,最早出现日期,最晚出现日期,出现次数,标准设备编号,标准设备名称,标准公司名称,匹配情况
0,Шөнө,Шөнө,NaN,效率_Sheet1,2022-01-09,2022-01-31,23,AA#0001,忽略,NaN,1.0
1,设备种类 Техникийн төрөл,设备种类 Техникийн төрөл,公司 Компани,效率_Sheet1,2026-04-19,2026-04-19,1,AA#0001,忽略,NaN,1.0
2,夜班 Шөнө,夜班 Шөнө,NaN,效率_Sheet1,2022-04-01,2026-04-19,152,AA#0001,忽略,NaN,1.0
3,CAT D8 #09,CAT D8 #09,NaN,效率_Sheet1,2019-10-05,2019-10-05,2,AA#0001,忽略,NaN,1.0
4,CAT D8 #10,CAT D8 #10,NaN,效率_Sheet1,2019-10-05,2019-10-05,2,AA#0001,忽略,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...
1783,TEREX TR100W #1211,TEREX TR100W #1211,Normount,效率_Sheet1,2025-03-25,2025-03-25,1,WT#1211,TEREX TR100W WT#1211,NORMOUNT,1.0
1784,WT #1211,WT #1211,Normount,效率_Sheet1,2025-03-23,2026-04-30,805,WT#1211,TEREX TR100W WT#1211,NORMOUNT,1.0
1785,WT#1211,WT#1211,NaN,Fuel_油耗信息,2025-03-23,2026-04-30,1003,WT#1211,TEREX TR100W WT#1211,NORMOUNT,1.0
1786,6683УЕА/ WT#1212 /XCMG / NXG5420/,WT#1212,NaN,Fuel_油耗信息,2026-04-23,2026-04-30,17,WT#1212,XCMG NXG5420 WT#1212,NORMOUNT,1.0


## 油耗匹配

In [61]:
# import data
fuel_file_path = 'Resources/Fuel_合并.xlsx'
fuel = pd.ExcelFile(fuel_file_path)
print(xlsx.sheet_names)

['Sheet1']


In [72]:
fuel_df = pd.read_excel(fuel_file_path, sheet_name=0)
fuel_df

,日期,班次,设备名称,设备编号,油品种类,油品消耗
0,2019-08-04,Night,TEREX TR100#56,HT0056,IC IC,476
1,2019-08-04,Night,TEREX TR100#58,HT0058,IC IC,462
2,2019-08-04,Night,TEREX TR100#59,HT0059,IC IC,451
3,2019-08-04,Night,TEREX TR100#60,HT0060,IC IC,454
4,2019-08-04,Night,TEREX TR100#61,HT0061,IC IC,502
...,...,...,...,...,...,...
222747,2026-04-30,Night,\nTOYOTA LANDCRUISER78 LV#1225,LV1225,Золотой /300007/,60
222748,2026-04-30,Night,9301УБР/ LV#1226 /TOYOTA /LAND 78/,LV1226,Золотой /300007/,30.84
222749,2026-04-30,Night,North Benz ST#137 /ST0137/,ST0137,Золотой /300007/,173.35
222750,2026-04-30,Night,North Benz ST#141,ST141,Золотой /300007/,350


In [73]:
fuel_summary = fuel_df.groupby(['油品种类']).agg(
    最早出现日期=('日期', 'min'),
    最晚出现日期=('日期', 'max'),
    出现次数=('油品消耗', 'count')
).reset_index() # 重置索引，让油品种类变回普通的列
# fuel_summary.to_excel('Resources/Fuel_摘要.xlsx', index=False)
# fuel_df['油品种类'].value_counts().reset_index(name='出现次数')
fuel_summary

,油品种类,最早出现日期,最晚出现日期,出现次数
0,IC IC,2019-08-04,2022-02-11,242
1,IC IC LLC,2023-04-24,2023-04-29,270
2,ICIC,2023-11-22,2024-01-05,231
3,ICIC LLC,2023-07-22,2023-07-22,1
4,Other,2023-12-01,2024-05-31,12477
5,PETROVIS,2019-10-12,2019-10-29,25
6,Petrovis,2019-09-03,2019-09-30,45
7,Pimary Energy,2019-08-13,2019-08-22,94
8,Primary,2019-10-01,2024-06-30,41563
9,Primary /300003/,2024-07-01,2026-04-24,13220


In [74]:
# import data
fuel_match_table_file_path = 'Resources/油品匹配表.xlsx'
fuel_match_table = pd.ExcelFile(fuel_match_table_file_path)
print(fuel_match_table.sheet_names)

['Sheet1']


In [75]:
fuel_match_table_df = pd.read_excel(fuel_match_table_file_path, sheet_name=0)
fuel_match_table_df

,油品种类,最早出现日期,最晚出现日期,出现次数,标准油品种类
0,IC IC,2019-08-04,2022-02-11,242,ICIC
1,IC IC LLC,2023-04-24,2023-04-29,270,ICIC
2,ICIC,2023-11-22,2024-01-05,231,ICIC
3,ICIC LLC,2023-07-22,2023-07-22,1,ICIC
4,Other,2023-12-01,2024-05-31,12477,Other
5,PETROVIS,2019-10-12,2019-10-29,25,Petrovis
6,Petrovis,2019-09-03,2019-09-30,45,Petrovis
7,Pimary Energy,2019-08-13,2019-08-22,94,Primary /300003/
8,Primary,2019-10-01,2024-06-30,41563,Primary /300003/
9,Primary /300003/,2024-07-01,2026-04-24,13220,Primary /300003/


In [76]:
fuel_merged = pd.merge(fuel_df, fuel_match_table_df[['油品种类','标准油品种类']], on='油品种类', how='left')
print(f'空值数量: {fuel_merged["标准油品种类"].isnull().sum()}')
fuel_merged['标准油品种类'].value_counts()

空值数量: 0


标准油品种类
Шунхлай /300008/             84496
Primary /300003/             55648
Синержи ойл /300009/         19721
Золотой /300007/             17409
НИК                          14475
Other                        12477
Сөүд монт форест /300012/     9050
Тод Өлзий /300010/            4053
Бусад /300001/                2966
Говь ойл газ /300013/         1607
ICIC                           744
Petrovis                        70
Петровис /300002/               36
Name: count, dtype: int64